## Imports

In [1]:
import os
import sys
sys.path.insert(0, os.path.abspath('/home/ec2-user/multimodal-rag'))

import nest_asyncio
nest_asyncio.apply()

import pandas as pd
import json
from tqdm.notebook import tqdm

# Allow notebooks to import from scripts/
REPO_ROOT = os.path.abspath("..")
sys.path.insert(0, os.path.join(REPO_ROOT, "scripts"))

import json
import pandas as pd
from pipeline import MMRAGPipeline

/Users/nicksmits/miniforge3/envs/GenAI/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Configurations

In [ ]:
SEED = 19
QUESTION_PATHS = {
    "FinancialReport" : "../data/raw/REAL-MM-RAG_FinReport/test/qas.jsonl",
    "TechReport" : "../data/raw/REAL-MM-RAG_TechReport/test/qas.jsonl",
    "TechSlides" : "../data/raw/REAL-MM-RAG_TechSlides/test/qas.jsonl"
}
RESULTS_PATH = "./results/baseline_ideal_rag_image"
model_id = "Qwen/Qwen3-VL-8B-Instruct"
checkpoint_path = "../Qwen3-VL-8B-Instruct"

## Load Data

In [3]:
test_indices = {}  # maps dataset name to set of example_index values for the test fold

for name, path in QUESTION_PATHS.items():
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            obj = json.loads(line)
            records.append({
                "example_index": obj["example_index"],
                "question": obj["question"],
                "answer": obj["answer"],
                "intended_img": obj.get("image_path"),
            })
    df = pd.DataFrame(records).set_index("example_index")
    train = df.sample(frac=0.8, random_state=SEED)
    test  = df.drop(train.index)
    test_indices[name] = set(test.index)
    print(f"{name}: {len(train)} train / {len(test)} test")

FinancialReport: 8 train / 2 test
TechReport: 8 train / 2 test
TechSlides: 8 train / 2 test


In [ ]:
VLM_MODEL_ID   = "Qwen/Qwen3-VL-8B-Instruct"
VLM_CHECKPOINT = os.path.join(REPO_ROOT, "Qwen3-VL-8B-Instruct")

QUESTION_PATHS = {
    "FinancialReport": os.path.join(REPO_ROOT, "data/raw/REAL-MM-RAG_FinReport/test/qas.jsonl"),
    "TechReport":      os.path.join(REPO_ROOT, "data/raw/REAL-MM-RAG_TechReport/test/qas.jsonl"),
    "TechSlides":      os.path.join(REPO_ROOT, "data/raw/REAL-MM-RAG_TechSlides/test/qas.jsonl"),
}

OCR_EMBEDDINGS_DIR = os.path.join(REPO_ROOT, "scripts", "embeddings_ocr")
RESULTS_DIR        = os.path.join(REPO_ROOT, "experiments", "results", "ocr_rag")

TOP_K    = 3
SAMPLING = {"max_new_tokens": 4096}

## Initialize Pipeline

In [5]:
pipeline = MMRAGPipeline(
    mode="ocr_rag",
    vlm_model_id=VLM_MODEL_ID,
    vlm_checkpoint_path=VLM_CHECKPOINT,
    ocr_retriever_persist_dir=OCR_EMBEDDINGS_DIR,
    top_k=TOP_K,
)

Initializing MMRAGPipeline | mode=ocr_rag | device=mps
--- Local model found at: /Users/nicksmits/Library/CloudStorage/OneDrive-Personal/CMU/GenAI/multimodal-rag/Qwen3-VL-2B-Instruct ---
  Import error: No module named 'triton'


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7413.74it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## Ingest Datasets

Index all three test splits into ChromaDB using OCR-extracted text. This only needs to run once — embeddings are persisted to disk and reloaded automatically on subsequent runs.

In [6]:
for name, qas_path in QUESTION_PATHS.items():
    print(f"\nIngesting {name}...")
    pipeline.ingest(qas_path)

print("\n✅ All datasets ingested.")


Ingesting FinancialReport...


Indexing OCR pages: 100%|██████████| 10/10 [00:00<00:00, 2691.07it/s]



Ingesting TechReport...


Indexing OCR pages: 100%|██████████| 10/10 [00:00<00:00, 3372.71it/s]



Ingesting TechSlides...


Indexing OCR pages: 100%|██████████| 10/10 [00:00<00:00, 5573.83it/s]


✅ All datasets ingested.


## Inference

In [7]:
os.makedirs(RESULTS_DIR, exist_ok=True)

for name, qas_path in QUESTION_PATHS.items():
    output_path = os.path.join(RESULTS_DIR, f"ocr_rag_{name}.json")
    print(f"\n{'='*60}")
    print(f"Running inference: {name}")
    print(f"{'='*60}")
    pipeline.run_inference(
        qas_jsonl_path=qas_path,
        output_path=output_path,
        example_indices=test_indices[name],
        sampling_params=SAMPLING,
    )

print("\n✅ Inference complete. Results saved to:", RESULTS_DIR)


Running inference: FinancialReport

Running inference on 2 examples | mode=ocr_rag
  dataset : /Users/nicksmits/Library/CloudStorage/OneDrive-Personal/CMU/GenAI/multimodal-rag/data/raw/REAL-MM-RAG_FinReport/test/qas.jsonl
  output  : /Users/nicksmits/Library/CloudStorage/OneDrive-Personal/CMU/GenAI/multimodal-rag/experiments/results/ocr_rag/ocr_rag_FinancialReport.json

  ERROR on example 2: probability tensor contains either `inf`, `nan` or element < 0

Saved 2 predictions → /Users/nicksmits/Library/CloudStorage/OneDrive-Personal/CMU/GenAI/multimodal-rag/experiments/results/ocr_rag/ocr_rag_FinancialReport.json

Running inference: TechReport

Running inference on 2 examples | mode=ocr_rag
  dataset : /Users/nicksmits/Library/CloudStorage/OneDrive-Personal/CMU/GenAI/multimodal-rag/data/raw/REAL-MM-RAG_TechReport/test/qas.jsonl
  output  : /Users/nicksmits/Library/CloudStorage/OneDrive-Personal/CMU/GenAI/multimodal-rag/experiments/results/ocr_rag/ocr_rag_TechReport.json


Saved 2 predic

## Evaluation

In [ ]:
# Run evaluate.py from the repo root to compute BLEU, ROUGE, BERTScore, and LLM-as-judge:
#
#   python scripts/evaluate.py experiments/results/ocr_rag/*.json \
#       --output experiments/results/ocr_rag/metrics_summary.json
#
# Set OPENAI_API_KEY in your environment to also enable LLM-as-judge scoring.